# 05 — Phase 2 Compliance Demo

**Purpose:** Run the compliance agent on synthetic test reports. Shows detected violations, retrieved regulations, and final decision. No UI — backend only.

**Design:** Rule engine (deterministic) → RAG explainer (retrieve + optional LLM) → hybrid decision → structured JSON.

Run from **project root** so `src` and `compliance_agent` are importable.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd() if Path.cwd().name != 'notebooks' else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Load .env from project root so GROQ_API_KEY / OPENAI_API_KEY are set
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
except ImportError:
    pass

from compliance_agent import run
from tests.fixtures.synthetic_reports import (
    CLEAN_REPORT,
    PRIVACY_VIOLATION_REPORT,
    OFF_LABEL_REPORT,
    AE_WITHOUT_REPORTING_REPORT,
    COMPARATIVE_CLAIM_REPORT,
    ALL_CASES,
)

import json

# LLM: set GROQ_API_KEY or OPENAI_API_KEY (e.g. in .env). Agent loads .env automatically.
USE_LLM = bool(__import__("os").environ.get("GROQ_API_KEY") or __import__("os").environ.get("OPENAI_API_KEY"))
print(f"Project root: {PROJECT_ROOT}")
print(f"Use LLM: {USE_LLM} (set GROQ_API_KEY or OPENAI_API_KEY for explanations)")

Project root: /Users/kumar/Documents/projects/new projects/Pharmaceutical Regulatory Compliance Agent
Use LLM: True (set GROQ_API_KEY or OPENAI_API_KEY for explanations)


## Run agent on each test case

For each synthetic report we show: **compliance_status**, **violations** (type, severity, evidence), and **retrieved_regulations** (when index exists and violations present).

In [2]:
for name, text, expected in ALL_CASES:
    result = run(text, top_k=3, use_llm=USE_LLM)
    status = result["compliance_status"]
    violations = result["violations"]
    print(f"\n{'='*60}")
    print(f"Case: {name} | Expected: {expected} | Got: {status}")
    print(f"Violations: {len(violations)}")
    for v in violations:
        print(f"  - [{v['severity']}] {v['type']}: {v['evidence'][:80]}...")
        if v.get('explanation'):
            print(f"    Explanation: {v['explanation'][:100]}...")
    if result.get("retrieved_regulations"):
        print(f"Retrieved regulations: {len(result['retrieved_regulations'])} chunks")
    print()


Case: clean | Expected: PASS | Got: PASS
Violations: 0



2026-02-06 01:16:10.619829: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):



Case: privacy | Expected: FAIL | Got: FAIL
Violations: 1
  - [critical] Patient Privacy Violation: PII/PHI detected: phone number, email, patient/person name...
    Explanation: Explanation unavailable (openai not installed). Review retrieved regulations below....
Retrieved regulations: 3 chunks


Case: off_label | Expected: FAIL | Got: FAIL
Violations: 1
  - [high] Off-Label Promotion: Promotional phrasing suggesting indication: effective for, recommended for. Veri...
    Explanation: Explanation unavailable (openai not installed). Review retrieved regulations below....
Retrieved regulations: 3 chunks


Case: ae_without_reporting | Expected: FAIL | Got: FAIL
Violations: 2
  - [critical] Patient Privacy Violation: PII/PHI detected: patient/person name...
    Explanation: Explanation unavailable (openai not installed). Review retrieved regulations below....
  - [high] Mandatory Reporting Required: Mandatory/expedited reporting flags present: hospitalization. Escalation require...
    E

## Full JSON output (one example: privacy violation)

In [3]:
result = run(PRIVACY_VIOLATION_REPORT, top_k=3, use_llm=USE_LLM)
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "compliance_status": "FAIL",
  "violations": [
    {
      "type": "Patient Privacy Violation",
      "severity": "critical",
      "evidence": "PII/PHI detected: phone number, email, patient/person name",
      "rule_id": "privacy",
      "regulatory_basis": "ICH",
      "explanation": "Explanation unavailable (openai not installed). Review retrieved regulations below.",
      "suggested_fix": "Redact or revise the flagged content and re-run compliance check."
    }
  ],
  "retrieved_regulations": [
    {
      "source": "ICH",
      "text": "1. Patient Details \n• Initials  \n• Other relevant identifier (p atient number, for example) \n• Gender \n• Age, age category (e.g., adolescent, adult, elderly), or date of birth \n• Concomitant conditions \n• Medical history \n• Relevant family history"
    },
    {
      "source": "ICH",
      "text": "5. GOOD CASE MANAGEMENT PRACTICES \nAccurate, complete, and bona fide inform ation is very important for MAHs and \nregulatory agencies ide